<a href="https://colab.research.google.com/github/jayaramanp/cNLP4QoL/blob/dev/cNLP4QoL_workbook3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/drmuskangarg/MultiWD.git
%cd MultiWD/
!ls

fatal: destination path 'MultiWD' already exists and is not an empty directory.
/content/MultiWD
README.md  src.ipynb  test_data.csv  train_data.csv


In [2]:
import pandas as pd
train_df = pd.read_csv("train_data.csv")
train_df

,text,Spiritual,Physical,Intellectual,Social,Vocational,Emotional
0,I feel like my life is spiraling downwards get...,0,0,0,1,0,1
1,I say sorry way too much. I get things said to...,0,0,0,0,0,1
2,It's like I'm always sad because I don't know ...,0,1,0,1,0,0
3,You know this feeling when there are people ar...,0,0,0,1,0,0
4,"A few months ago (possibly more, my perception...",0,1,0,1,0,0
...,...,...,...,...,...,...,...
2619,I've had this since I was a kid. When I'm goin...,0,0,0,0,0,1
2620,Ive always been denying that I might have depr...,0,0,0,0,0,1
2621,I've talked with her for over a year. I develo...,0,1,0,1,0,1
2622,"It will get better for you, the person with mo...",0,0,0,1,0,1


In [3]:
test_df = pd.read_csv("test_data.csv")
test_df['text']

,text
0,I don‚Äôt understand how I‚Äôm feeling and all...
1,I hate every single thing about myself and I d...
2,"So today in particular was a very bad day, I f..."
3,My dad has had a long history of cheating on m...
4,I'm not sure which is worse: to be completely ...
...,...
652,Why does everyone think that it's just a tempo...
653,I hate myself so much It hurts I hate my body ...
654,I've been lying here for hours just doing noth...
655,"Married to a great guy, two great kids, a job ..."


In [4]:
test_df

,text,Spiritual,Physical,Intellectual,Social,Vocational,Emotional
0,I don‚Äôt understand how I‚Äôm feeling and all...,0,0,0,1,1,1
1,I hate every single thing about myself and I d...,0,0,0,1,0,0
2,"So today in particular was a very bad day, I f...",0,0,0,1,0,1
3,My dad has had a long history of cheating on m...,0,1,0,1,1,1
4,I'm not sure which is worse: to be completely ...,0,0,0,1,0,0
...,...,...,...,...,...,...,...
652,Why does everyone think that it's just a tempo...,0,0,0,1,0,1
653,I hate myself so much It hurts I hate my body ...,0,0,0,1,0,1
654,I've been lying here for hours just doing noth...,0,1,1,0,0,0
655,"Married to a great guy, two great kids, a job ...",0,1,1,1,1,1


LLM-Based Prompting

In [5]:
import openai
import json
from tenacity import retry, stop_after_attempt, wait_random_exponential

# Set API key
openai.api_key = "your-api-key-here"


In [6]:

# Define system and few-shot prompt
SYSTEM_PROMPT = """You are an expert clinical NLP system specialized in identifying quality-of-life (QoL) signals in therapist–patient conversations.

QoL Domains:
1. Physical Well-Being: Pain, mobility, strength, physical limitations
2. Emotional Well-Being: Mood, anxiety, depression, psychological distress
3. Social Functioning: Relationships, social activities, loneliness
4. Activities of Daily Living: Self-care, work, household tasks
5. Sleep Quality: Sleep disturbance, insomnia, daytime sleepiness
6. Pain: Pain intensity, location, duration, impact
7. Fatigue: Tiredness, lack of energy, difficulty concentrating

Task: For each conversation turn, identify all applicable QoL domains.

Response Format: Return a JSON object with:
{
  "qol_domains": ["Domain1", "Domain2"],
  "confidence": [0.95, 0.88],
  "reasoning": "Brief explanation of why these domains were selected"
}

Be conservative: Only label domains explicitly mentioned or clearly implied by context.
"""

FEW_SHOT_EXAMPLES = """
Example 1:
[PATIENT]: "I'm exhausted all the time, can barely get out of bed in the morning."
Response:
{
  "qol_domains": ["Fatigue", "Emotional Well-Being"],
  "confidence": [0.98, 0.75],
  "reasoning": "Patient explicitly mentions exhaustion (Fatigue) and implied depression/low mood (Emotional Well-Being)"
}

Example 2:
[CLINICIAN]: "Let's discuss your medication side effects."
Response:
{
  "qol_domains": [],
  "confidence": [],
  "reasoning": "Clinician turn with procedural content; no direct QoL signal"
}

Example 3:
[PATIENT]: "My knees hurt when I climb stairs, so I've stopped going to the gym with my friends."
Response:
{
  "qol_domains": ["Physical Well-Being", "Pain", "Social Functioning"],
  "confidence": [0.95, 0.95, 0.90],
  "reasoning": "Explicit pain and mobility limitation (Physical/Pain); implied social withdrawal (Social Functioning)"
}
"""

@retry(wait=wait_random_exponential(min=1, max=60), stop=stop_after_attempt(3))
def call_llm_qol_tagger(turn_text, model="gpt-4", temperature=0.1):
    """Call LLM API with retry logic for rate limiting"""

    user_message = f"""Classify this conversation turn:

{turn_text}

Response:"""

    response = openai.ChatCompletion.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": FEW_SHOT_EXAMPLES + "\n" + user_message}
        ],
        temperature=temperature,
        max_tokens=200
    )

    return response.choices[0].message.content

def parse_llm_response(response_text):
    """Parse LLM JSON response; handle failures gracefully"""
    try:
        # Extract JSON from response
        json_start = response_text.find("{")
        json_end = response_text.rfind("}") + 1
        json_str = response_text[json_start:json_end]
        result = json.loads(json_str)
        return result
    except (json.JSONDecodeError, ValueError) as e:
        print(f"Failed to parse LLM response: {e}")
        return {"qol_domains": [], "confidence": [], "reasoning": "Parse error"}

# Test on sample turns
test_turns = [
    "[PATIENT]: I haven't slept well in weeks. My back aches from lying down so much.",
    "[CLINICIAN]: How is your appetite?",
    "[PATIENT]: It's terrible. I have no energy and I'm so sad."
]

llm_predictions = []
for turn in test_turns:
    print(f"\nTurn: {turn[:60]}...")
    response = call_llm_qol_tagger(turn)
    parsed = parse_llm_response(response)
    llm_predictions.append(parsed)
    print(f"Predicted QoL Domains: {parsed['qol_domains']}")
    print(f"Confidence: {parsed['confidence']}")


Turn: [PATIENT]: I haven't slept well in weeks. My back aches from...


RetryError: RetryError[<Future at 0x7964a2087c50 state=finished raised APIRemovedInV1>]